# Corte 3 · Procesamiento distribuido con MapReduce (Google Colab)

**Proyecto de Aula Data Warehouse y Big Data · Politécnico Grancolombiano**

Este notebook monta un Hadoop 3.3.6 pseudo-distribuido (HDFS + YARN reales) dentro de la máquina virtual de Colab y ejecuta los seis jobs MapReduce del proyecto:

| Job | Tipo | Pregunta que responde |
|---|---|---|
| 1 · `job1_terminos` | Conteo con combiner | ¿Qué palabras anticipan una reseña negativa? (modelo Naive Bayes) |
| 2a · `job2a_join` | Reduce-side join | Cruce de reseñas (texto) con entregas de la bodega |
| 2b · `job2b_temas` | Agregación | ¿De qué se quejan los clientes y cuánto pesa la entrega? |
| 3a · `job3a_centroides` | Agregación con combiner | Coordenadas de 19 mil códigos postales a partir de 1 millón de puntos |
| 3b · `job3b_distancias` | Map-side join (caché distribuida) | ¿Cuánto explica la distancia real el retraso? |
| 4 · `job4_clasificar` | Scoring + matriz de confusión | ¿Qué tan bien se clasifica una reseña solo por su texto? |

Ejecuta **Entorno de ejecución → Ejecutar todas** (≈8 minutos).

## 1. Clonar el repositorio y reconstruir la bodega (Corte 2)

In [ ]:
import os, subprocess
REPO_URL = os.environ.get("REPO_URL", "https://github.com/juanjose1099322-lang/DWH.git")
REPO_DIR = "/content/olist-dw-bigdata"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
!cp -n config/.env.example config/.env
!python etl/etl_olist.py
!python etl/ejecutar_sql.py sql/02_cubos_rolap.sql

## 2. Instalar Hadoop pseudo-distribuido

Java 11, Hadoop 3.3.6, configuración de `core-site.xml`, `hdfs-site.xml` (bloques de 16 MB), `mapred-site.xml` y `yarn-site.xml`, formateo del NameNode e inicio de los demonios. Todo queda en `localhost` (cadena estándar local `hdfs://localhost:9000`).

In [ ]:
!bash scripts/setup_hadoop.sh

## 3. Preparar insumos y cargarlos en HDFS

`preparar_insumos.py` deja cada reseña en una sola línea (Hadoop parte por líneas), exporta las entregas desde la bodega y copia la geolocalización cruda.

In [ ]:
!python mapreduce/preparar_insumos.py

In [ ]:
%%bash
source /etc/profile.d/hadoop.sh
hdfs dfs -mkdir -p /olist/raw /olist/mr
hdfs dfs -put -f data/hdfs_in/* /olist/raw/
hdfs dfs -ls -h /olist/raw
hdfs fsck /olist/raw/geolocalizacion.csv -files -blocks | grep -E "blk_|Total blocks"
hdfs dfs -cat /olist/raw/resenas.tsv | head -3

## 4. Ejecutar los seis jobs en YARN

El script guarda logs, contadores, tiempos y métricas del ResourceManager en `evidencias/`.

In [ ]:
%%bash
source /etc/profile.d/hadoop.sh
bash mapreduce/run_hadoop.sh 2>&1 | grep -E "Running job|Job job_.*completed|Map input records|Reduce output records|OK |DIFERENTE|Listo"


## 5. Monitoreo de YARN

In [ ]:
%%bash
source /etc/profile.d/hadoop.sh
yarn application -list -appStates FINISHED
echo; cat evidencias/tiempos_yarn.tsv
echo; curl -s http://localhost:8088/ws/v1/cluster/metrics | python3 -m json.tool | head -30

## 6. Leer resultados desde HDFS

In [ ]:
%%bash
source /etc/profile.d/hadoop.sh
echo "== Temas de las reseñas =="; hdfs dfs -cat /olist/mr/job2b_temas/part-* | sort -t$'\t' -k2,2nr
echo "== Distancia real vs entrega =="; hdfs dfs -cat /olist/mr/job3b_distancias/part-* | grep -v RUTA | sort
echo "== Matriz de confusión =="; hdfs dfs -cat /olist/mr/job4_clasificar/part-* | sort

## 7. Integración: salidas de MapReduce → bodega → tablero

In [ ]:
!python mapreduce/cargar_resultados_dw.py
!python etl/exportar_dashboard.py
from IPython.display import HTML
HTML(open("dashboard/tablero_fragmento.html", encoding="utf-8").read())

## 8. Contraste de rendimiento DW vs MapReduce (opcional, ≈3 min)

In [ ]:
!FACTORES=1,5 python mapreduce/benchmark_dw_vs_mr.py | grep factor

## 9. Versionar evidencias en GitHub

Guarda un token personal en **Secretos de Colab** con el nombre `GITHUB_TOKEN` (nunca en el notebook).

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!git config user.email "equipo@poli.edu.co" && git config user.name "Equipo DWH"
!git checkout -B feature/corte3-evidencias-colab
!git add evidencias dashboard && git commit -m "Agrega evidencias de Hadoop en Colab (Corte 3)" || true
!git push -u https://{token}@github.com/juanjose1099322-lang/DWH.git feature/corte3-evidencias-colab